# Fine-Tuning an LLM with Synthetic Training Data from Dataframer

You want to fine-tune a language model to write in a specific style, but you only have **2 examples**. That's not enough for fine-tuning — most runs need at least 50–100 diverse training samples to learn a style without memorizing content.

In this notebook, we use [Dataframer](https://dataframer.ai) to turn those 2 examples into 100 diverse training samples, then fine-tune [Llama 3 8B](https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct) on [Together AI](https://together.ai), and evaluate the results.

**The task:** Write technical explanations that follow 8 specific style criteria:
1. Opens by stating the end result first, then works backward
2. Each paragraph introduces exactly one new mechanism or idea
3. Uses "the trick is" or "the problem is" as a transition
4. The single most striking number gets its own short sentence
5. Explains by contrasting with a simpler/broken version
6. Final paragraph is a single sentence
7. 3–5 paragraphs
8. No analogies or metaphors — only literal descriptions

### Prerequisites

You'll need:
- A [Dataframer API key](https://app.dataframer.ai) (Account → Keys → Copy API Key)
- A [Together AI API key](https://api.together.xyz/settings/api-keys)

In [ ]:
%%capture
%pip install --upgrade pydataframer together==2.5.0 pyyaml tenacity

In [ ]:
import os

try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Running in Google Colab. Loading secrets from Colab userdata...")
    os.environ["DATAFRAMER_API_KEY"] = userdata.get("DATAFRAMER_API_KEY")
    os.environ["TOGETHER_API_KEY"] = userdata.get("TOGETHER_API_KEY")
else:
    print("Not in Colab. Relying on local environment variables...")
    ## Optional: You can use python-dotenv
    # from dotenv import load_dotenv
    # load_dotenv() 

In [ ]:
import io
import json
import tempfile
import textwrap
import zipfile
from datetime import datetime
import yaml
from dataframer import Dataframer
from tenacity import retry, retry_if_result, stop_never, wait_fixed
import asyncio
from together import AsyncTogether

async_client = AsyncTogether(api_key=os.environ["TOGETHER_API_KEY"])
df_client = Dataframer(api_key=os.environ["DATAFRAMER_API_KEY"])

## Part 1: Generate Training Data with Dataframer

We start with just 2 hand-written examples that demonstrate the style we want. These are question–answer pairs about how everyday technology works.

In [4]:
SEED_EXAMPLES = [
    {
        "question": "How does your phone's GPS figure out where you are?",
        "answer": "Your phone knows its location by measuring how long radio signals take to arrive from satellites. There are at least 24 of them orbiting Earth at any given time, and your phone listens to whichever ones are overhead.\n\nThe problem is that radio waves travel at the speed of light, so the time differences are absurdly small. Each satellite stamps its signal with the exact time it was sent. Your phone compares that to when the signal arrived. The difference tells you the distance to that satellite. One satellite gives you a sphere of possible locations. Two satellites narrow it to a circle. Three give you two points. Four pin you down to one.\n\nThe trick is that your phone's clock is garbage compared to the atomic clocks on the satellites. A microsecond of drift means 300 meters of error. So the fourth satellite isn't really for position \u2014 it's for correcting your phone's clock. The math solves for four unknowns: x, y, z, and time.\n\n150 nanoseconds. That's the timing precision the system needs to get you within a meter.\n\nFour satellites, four unknowns, and a clock correction disguised as a position fix."
    },
    {
        "question": "How does a microwave heat food?",
        "answer": "A microwave oven heats food by making water molecules spin. There's a component called a magnetron inside the oven that generates electromagnetic radiation at a specific frequency, and that radiation shoves water molecules back and forth.\n\nThe problem is that water molecules are lopsided \u2014 one end is slightly positive, the other slightly negative. When the microwave field flips direction, every water molecule tries to reorient itself to line up with the new field. The field flips 2.45 billion times per second. The molecules can't keep up, and the friction from all that failed rotation becomes heat.\n\nThe trick is that this only works on molecules that have that lopsided charge distribution. A completely dry ceramic plate won't heat up at all. Oil heats poorly because its molecules are less polar than water. Your soup gets hot while the bowl stays cool because the microwaves pass through the ceramic and dump their energy into the water in the soup.\n\n2.45 billion flips per second. That's the frequency, and it was chosen not because water absorbs it best, but because it penetrates food a few centimeters deep instead of being absorbed at the surface.\n\nAll the heat comes from water molecules that can't spin fast enough."
    },
]

for i, ex in enumerate(SEED_EXAMPLES, 1):
    print(f"--- Seed {i} ---")
    print(f"Q: {ex['question']}")
    print(f"A: {ex['answer'][:200]}...\n")

--- Seed 1 ---
Q: How does your phone's GPS figure out where you are?
A: Your phone knows its location by measuring how long radio signals take to arrive from satellites. There are at least 24 of them orbiting Earth at any given time, and your phone listens to whichever on...

--- Seed 2 ---
Q: How does a microwave heat food?
A: A microwave oven heats food by making water molecules spin. There's a component called a magnetron inside the oven that generates electromagnetic radiation at a specific frequency, and that radiation ...



### Upload seed data to Dataframer

We package our 2 examples as a JSON file and upload them as a seed dataset.

In [5]:
zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, "w") as zf:
    zf.writestr("seed_examples.json", json.dumps(SEED_EXAMPLES, indent=2))
zip_buffer.seek(0)

dataset = df_client.dataframer.seed_datasets.create_from_zip(
    name=f"sft_seed_data_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
    description="2 example explanations for SFT style transfer",
    zip_file=zip_buffer,
)
dataset_id = dataset.id
print(f"Dataset uploaded: {dataset_id}")
print(f"Files: {dataset.file_count}")

Dataset uploaded: 0059247b-5723-4d75-8ad4-c3b8d622fa35
Files: 1


### Create a spec with generation objectives

We tell Dataframer what the style criteria are via `generation_objectives`. Dataframer analyzes the seed data, learns the patterns, and creates generation rules that combine both the observed patterns and our explicit criteria.

In [6]:
GENERATION_OBJECTIVES = """All of the following criteria must be met by every generated sample:
1. Opens by stating the end result first, then works backward
2. Each paragraph introduces exactly one new mechanism or idea
3. Uses "the trick is" or "the problem is" as a transition
4. The single most striking number gets its own short sentence
5. Explains by contrasting with a simpler/broken version
6. Final paragraph is a single sentence
7. 3-5 paragraphs
8. No analogies or metaphors - only literal descriptions"""

spec = df_client.dataframer.specs.create(
    dataset_id=dataset_id,
    name=f"sft_explanations_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
    generation_objectives=GENERATION_OBJECTIVES,
    generate_distributions=True,
    generate_conditional_distributions=True,
    extrapolate_values=True,
)
spec_id = spec.id
print(f"Spec creation started: {spec_id}")
print("Waiting for analysis to complete (usually 1-2 minutes)...")


@retry(wait=wait_fixed(10), retry=retry_if_result(lambda r: r.status == "PROCESSING"), stop=stop_never)
def poll_spec(spec_id):
    return df_client.dataframer.specs.retrieve(spec_id=spec_id)


spec_result = poll_spec(spec_id)
assert spec_result.status == "SUCCEEDED", f"Spec failed: {spec_result.error}"
print(f"Spec ready!")

Spec creation started: c178f62e-cb41-42e4-8d4c-e5f13e8eefb6
Waiting for analysis to complete (usually 1-2 minutes)...
Spec ready!


In [7]:
config = yaml.safe_load(spec_result.content_yaml)
spec_data = config["spec"]

print("Requirements (excerpt):")
print(textwrap.fill(spec_data["requirements"][:500], width=100))
print()
print("Data property variations:")
for prop in spec_data["data_property_variations"]:
    print(f"  {prop['property_name']}: {len(prop['property_values'])} values")
    for v in prop["property_values"][:5]:
        print(f"    - {v}")
    if len(prop["property_values"]) > 5:
        print(f"    ... and {len(prop['property_values']) - 5} more")

Requirements (excerpt):
The data point is a question-and-answer pair. The question is phrased as 'How does [X]
[do/work/function]?' The answer opens by stating the end result or observable output first in one or
two sentences, then works backward through the mechanism that produces it. Each paragraph introduces
exactly one new mechanism, constraint, or idea with no overlapping concepts between paragraphs. The
answer uses 'The problem is' as a transition to introduce a constraint or complication, and 'The
trick is' as a

Data property variations:
  Scientific or technical domain of the topic: 20 values
    - electromagnetism and wave physics
    - signal processing and communications
    - molecular biology and biochemistry
    - thermodynamics and heat transfer
    - computing hardware and digital logic
    ... and 15 more
  Specific topic being explained: 30 values
    - how a microwave heats food
    - how GPS determines location
    - how MRI machines produce images
    - how CRISPR

### Generate 100 training samples

Now we run generation. Dataframer will produce 100 diverse question–answer pairs, all following the style patterns from our 2 seeds and the generation objectives.

In [11]:
run = df_client.dataframer.runs.create(
    spec_id=spec_id,
    number_of_samples=100,
    generation_model="anthropic/claude-sonnet-4-6-thinking",
    outline_model="anthropic/claude-sonnet-4-6-thinking",
    skip_outline=True,
)
run_id = run.id
print(f"Generation started: {run_id}")
print("Generating 100 samples (this may take several minutes)...")


@retry(wait=wait_fixed(15), retry=retry_if_result(lambda r: r.status not in ("SUCCEEDED", "FAILED")), stop=stop_never)
def poll_run(run_id):
    r = df_client.dataframer.runs.retrieve(run_id=run_id)
    print(f"  {r.samples_completed} completed, {r.samples_failed} failed", flush=True)
    return r


run_result = poll_run(run_id)
assert run_result.status == "SUCCEEDED"
print(f"\nDone! {run_result.samples_completed} samples generated, {run_result.samples_failed} failed.")

Generation started: afdec095-f4d6-4c40-8051-a7761b551902
Generating 100 samples (this may take several minutes)...
  0 completed, 0 failed
  5 completed, 0 failed
  16 completed, 0 failed
  50 completed, 0 failed
  100 completed, 0 failed

Done! 100 samples generated, 0 failed.


### Download and inspect generated samples

In [12]:
import requests


@retry(wait=wait_fixed(3), retry=retry_if_result(lambda r: r.download_url is None), stop=stop_never)
def poll_download(run_id):
    return df_client.dataframer.runs.files.download_all(run_id=run_id)


download = poll_download(run_id)
zip_bytes = requests.get(download.download_url).content

with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
    for name in zf.namelist():
        if name.endswith(".json") and "metadata" not in name:
            generated_samples = json.loads(zf.read(name))
            break

print(f"Downloaded {len(generated_samples)} samples\n")

for i, sample in enumerate(generated_samples[:3]):
    print(f"=== Sample {i+1} ===")
    print(f"Q: {sample['question']}")
    print(f"A: {sample['answer'][:300]}...")
    print()

Downloaded 100 samples

=== Sample 1 ===
Q: How does a lithium-ion battery achieve energy storage?
A: A lithium-ion battery stores energy by moving lithium ions between two electrodes, and it releases that energy by letting them move back. The electrical current available to a device is a direct consequence of that ionic movement through an electrolyte combined with electron flow through an external...

=== Sample 2 ===
Q: How does a seismograph record ground motion?
A: A seismograph records ground motion by keeping a suspended mass stationary while the ground beneath it moves. The difference in position between the mass and the moving frame gets converted into a continuous trace that encodes the timing, direction, and amplitude of seismic waves.

The problem is th...

=== Sample 3 ===
Q: How does neural network training work?
A: A neural network learns by repeatedly adjusting the numerical weights on its internal connections until its output errors shrink to an acceptable level. Every

## Part 2: Fine-Tune with Together AI

We now take the 100 generated samples and fine-tune Llama 3 8B Instruct on Together AI. The training data needs to be in JSONL chat format.

In [13]:
import together

together_client = together.Together()

# Convert to JSONL chat format
jsonl_path = tempfile.mktemp(suffix=".jsonl")
with open(jsonl_path, "w") as f:
    for s in generated_samples:
        line = json.dumps({"messages": [
            {"role": "user", "content": s["question"]},
            {"role": "assistant", "content": s["answer"]},
        ]})
        f.write(line + "\n")

print(f"Training data: {len(generated_samples)} examples")

# Upload to Together
file_resp = together_client.files.upload(jsonl_path, check=True)
print(f"Uploaded: {file_resp.id}")

Training data: 100 examples


Validating file: 100 lines [00:00, 71319.57 lines/s]
Uploading file tmpw9zo2_zp.jsonl: 100%|██████████| 168k/168k [00:00<00:00, 519kB/s]


Uploaded: file-8602bf30-61fc-4172-98c0-4f80eb91b8e2


In [ ]:
import time

ft_job = together_client.fine_tuning.create(
    training_file=file_resp.id,
    model="meta-llama/Meta-Llama-3-8B-Instruct",
    n_epochs=20,
    learning_rate=1e-5,
    lora=False,
    suffix="explain-style",
)
print(f"Fine-tuning job: {ft_job.id}")
print(f"Base model: meta-llama/Meta-Llama-3-8B-Instruct")
print(f"Full fine-tune, lr=1e-5, 20 epochs")
print("Training (this usually takes a few minutes)...")

while True:
    status = together_client.fine_tuning.retrieve(ft_job.id)
    print(f"  Status: {status.status}", flush=True)
    if status.status in ("completed", "failed", "cancelled", "error"):
        break
    time.sleep(60)

assert status.status == "completed"
finetuned_model = status.model_output_name
print(f"\nFine-tuned model: {finetuned_model}")

### Deploy the model

Create a Together endpoint to serve the fine-tuned model.

In [21]:
endpoint = together_client.endpoints.create(
    model=finetuned_model,
    hardware="1x_nvidia_h100_80gb_sxm",
    display_name="explain-style-eval",
    autoscaling={"min_replicas": 1, "max_replicas": 1},
)
print(f"Endpoint: {endpoint.id}")
print(f"Inference name: {endpoint.name}")
print("Waiting for endpoint to be ready...")

while True:
    ep = together_client.endpoints.retrieve(endpoint.id)
    if ep.state == "STARTED":
        break
    time.sleep(10)

model_name = endpoint.name
print(f"Endpoint ready: {model_name}")

Endpoint: endpoint-ce98051b-621a-4c51-b47c-31d259f23c42
Inference name: preetamj/Meta-Llama-3-8B-Instruct-explain-style-0cace4a9-2c8529d9
Waiting for endpoint to be ready...
Endpoint ready: preetamj/Meta-Llama-3-8B-Instruct-explain-style-0cace4a9-2c8529d9


## Part 3: Evaluate

We evaluate the fine-tuned model on 16 held-out questions spanning technology, math, and philosophy — none of which appeared in the training data.

In [22]:
EVAL_QUESTIONS = [
    "How does a touchscreen know where your finger is?",
    "How does WiFi get through walls?",
    "How does noise-canceling work in headphones?",
    "How does your phone charge wirelessly?",
    "How does a QR code store information?",
    "How does facial recognition unlock your phone?",
    "How does autocorrect predict what you meant to type?",
    "How does a Bluetooth speaker connect to your phone?",
    "What does it actually mean to take the derivative of a function?",
    "Why does multiplying two negative numbers give a positive?",
    "What is the intuition behind eigenvalues?",
    "Why does 0.1 + 0.2 not equal 0.3 in computers?",
    "What is the problem of free will?",
    "What is the hard problem of consciousness?",
    "What does G\u00f6del's incompleteness theorem actually say?",
    "Why is the trolley problem interesting?",
]

print(f"{len(EVAL_QUESTIONS)} evaluation questions across 3 domains:")
print(f"  Technology (1-8), Math (9-12), Philosophy (13-16)")

16 evaluation questions across 3 domains:
  Technology (1-8), Math (9-12), Philosophy (13-16)


In [30]:
async def run_eval(model, questions):
    async def ask(q, i):
        print(f"  {i+1}/{len(questions)}: {q[:50]}...", flush=True)
        resp = await async_client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": q}],
            max_tokens=5000,
        )
        return {"question": q, "answer": resp.choices[0].message.content}

    return await asyncio.gather(*[ask(q, i) for i, q in enumerate(questions)])

results = await run_eval(model_name, EVAL_QUESTIONS)
print(f"\nGot {len(results)} responses")

  1/16: How does a touchscreen know where your finger is?...
  2/16: How does WiFi get through walls?...
  3/16: How does noise-canceling work in headphones?...
  4/16: How does your phone charge wirelessly?...
  5/16: How does a QR code store information?...
  6/16: How does facial recognition unlock your phone?...
  7/16: How does autocorrect predict what you meant to typ...
  8/16: How does a Bluetooth speaker connect to your phone...
  9/16: What does it actually mean to take the derivative ...
  10/16: Why does multiplying two negative numbers give a p...
  11/16: What is the intuition behind eigenvalues?...
  12/16: Why does 0.1 + 0.2 not equal 0.3 in computers?...
  13/16: What is the problem of free will?...
  14/16: What is the hard problem of consciousness?...
  15/16: What does Gödel's incompleteness theorem actually ...
  16/16: Why is the trolley problem interesting?...

Got 16 responses


In [31]:
# Show a few responses
for r in results[:3]:
    print(f"Q: {r['question']}")
    print(f"A: {r['answer'][:400]}...")
    print()

Q: How does a touchscreen know where your finger is?
A: A touchscreen determines the location of a touch by measuring tiny changes in electrical capacitance at precise grid intersections across its surface.

The problem is that the screen must resolve not just whether a touch occurred, but exactly where, across a surface that may have thousands of distinct sensing points. A resistive touchscreen solves this by physically pressing two conductive layers ...

Q: How does WiFi get through walls?
A: WiFi transmits data by encoding binary information onto radio waves and broadcasting those waves through the air. Your device's wireless card receives those waves and decodes the original data from them.

The problem is that walls are not perfectly transparent to radio waves. When a wave strikes a surface, some of its energy reflects back, some absorbs into the material, and some passes through. T...

Q: How does noise-canceling work in headphones?
A: Noise-canceling headphones reduce unwanted s

### Score against style criteria

We use an LLM to score each response against the 8 style criteria. Each criterion is scored as 1 (met) or 0 (not met) for each response.

In [32]:
CRITERIA = [
    "C1: Opens by stating the end result first, then works backward",
    "C2: Each paragraph introduces exactly one new mechanism or idea",
    'C3: Uses "the trick is" or "the problem is" as a transition phrase',
    "C4: The single most striking number gets its own short sentence",
    "C5: Explains by contrasting with a simpler or broken version",
    "C6: Final paragraph is a single sentence",
    "C7: 3-5 paragraphs total",
    "C8: No analogies or metaphors — only literal descriptions",
]

SCORING_PROMPT = """Score this response against each criterion. Reply with ONLY a JSON object mapping criterion labels (C1-C8) to 1 (met) or 0 (not met).

Criteria:
{criteria}

Question: {question}

Response:
{answer}

JSON:"""

async def score_response(result):
    prompt = SCORING_PROMPT.format(
        criteria="\n".join(CRITERIA),
        question=result["question"],
        answer=result["answer"],
    )
    resp = await async_client.chat.completions.create(
        model="deepseek-ai/DeepSeek-V3.1",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=32000,
    )
    text = resp.choices[0].message.content.strip()
    # Extract JSON from response
    start = text.find("{")
    end = text.rfind("}") + 1
    scores = json.loads(text[start:end])
    return scores


print("Scoring all responses...")
all_scores = await asyncio.gather(*[score_response(r) for r in results])
print("Done!")

Scoring all responses...
Done!


In [33]:
# Build results table
criteria_labels = [f"C{i}" for i in range(1, 9)]
question_names = [
    "Touchscreen", "WiFi", "Noise-canceling", "Wireless charging",
    "QR code", "Facial recognition", "Autocorrect", "Bluetooth",
    "Derivative", "Neg \u00d7 neg", "Eigenvalues", "0.1+0.2\u22600.3",
    "Free will", "Consciousness", "G\u00f6del", "Trolley",
]

# Print per-question scores
header = f"{'Question':<20} " + " ".join(f"{c:>3}" for c in criteria_labels) + "  Total"
print(header)
print("-" * len(header))

totals = {c: 0 for c in criteria_labels}
for name, scores in zip(question_names, all_scores):
    row_scores = [scores.get(c, 0) for c in criteria_labels]
    for c, s in zip(criteria_labels, row_scores):
        totals[c] += s
    row = f"{name:<20} " + " ".join(f"{'\u2713' if s else '\u2717':>3}" for s in row_scores)
    row += f"  {sum(row_scores)}/8"
    print(row)

print("-" * len(header))
total_row = f"{'TOTAL':<20} " + " ".join(f"{totals[c]:>3}" for c in criteria_labels)
grand_total = sum(totals.values())
total_row += f"  {grand_total}/{len(results) * 8}"
print(total_row)
print(f"\nOverall style score: {grand_total}/{len(results) * 8} ({100 * grand_total / (len(results) * 8):.0f}%)")

Question              C1  C2  C3  C4  C5  C6  C7  C8  Total
-----------------------------------------------------------
Touchscreen            ✓   ✓   ✓   ✓   ✓   ✓   ✓   ✓  8/8
WiFi                   ✗   ✗   ✓   ✗   ✗   ✗   ✗   ✗  1/8
Noise-canceling        ✓   ✓   ✓   ✓   ✓   ✓   ✓   ✓  8/8
Wireless charging      ✓   ✗   ✓   ✓   ✓   ✗   ✗   ✓  5/8
QR code                ✓   ✓   ✓   ✓   ✓   ✓   ✓   ✓  8/8
Facial recognition     ✓   ✗   ✓   ✓   ✓   ✓   ✗   ✓  6/8
Autocorrect            ✓   ✗   ✓   ✓   ✓   ✗   ✗   ✗  4/8
Bluetooth              ✓   ✓   ✓   ✓   ✓   ✗   ✗   ✓  6/8
Derivative             ✓   ✓   ✓   ✓   ✓   ✗   ✗   ✓  6/8
Neg × neg              ✓   ✗   ✓   ✓   ✗   ✗   ✗   ✗  3/8
Eigenvalues            ✗   ✗   ✓   ✓   ✗   ✗   ✗   ✓  3/8
0.1+0.2≠0.3            ✗   ✗   ✓   ✗   ✗   ✗   ✗   ✗  1/8
Free will              ✓   ✓   ✓   ✗   ✓   ✗   ✗   ✓  5/8
Consciousness          ✓   ✗   ✓   ✓   ✓   ✓   ✗   ✓  6/8
Gödel                  ✓   ✗   ✓   ✓   ✓   ✗   ✗   ✓  5/8
Trolley   

### Cleanup

Stop the Together endpoint when you're done to avoid ongoing charges.

In [34]:
together_client.endpoints.delete(endpoint.id)
print("Endpoint deleted")

Endpoint deleted


## Baseline: What If You Just Replicate the Seeds?

A natural question: do you really need Dataframer? What if you just duplicate the 2 seed examples 50 times each to get 100 training samples?

We ran this experiment with the same model, same hyperparameters, and same evaluation. The result: **catastrophic memorization**. The model learned the *content* of the two seeds, not the *style*. It regurgitates GPS and microwave facts for every question, often falling into repetition loops.

Here's what the baseline model produces when asked "How does a touchscreen know where your finger is?":

> *A touchscreen knows its location by measuring how long radio signals take to arrive from your finger. There's a component called a controller inside the screen that generates radio signals at a specific frequency, and your finger absorbs them at the speed of light.*
>
> *The problem is that radio waves travel at the speed of light, so the time differences are absurdly small. Each radio signal shoves water molecules back and forth...*
>
> *2.45 billion flips per second. That's the frequency, and it was chosen not because water absorbs it best, but because it penetrates food a few centimeters deep...*
>
> *(repeats for several more paragraphs)*

It's mixing GPS satellite content, microwave magnetron content, and repeating the same sentences in a loop. The model memorized the two training examples instead of learning the writing style.

| Training data | Style score | Content correct |
|---|---|---|
| 100 diverse samples (Dataframer) | 84% | 15/16 |
| 2 seeds \u00d7 50 copies | Degenerate | ~2-3/16 |

**Takeaway:** Diverse training examples are essential for style transfer. Replicating a handful of examples teaches the model what to say, not how to say it.

In [ ]:
# For reference: this is how the baseline training data was constructed.
# Do NOT use this approach — it leads to catastrophic memorization.

# import random
#
# baseline_samples = []
# for _ in range(50):
#     baseline_samples.append(SEED_EXAMPLES[0])
# for _ in range(50):
#     baseline_samples.append(SEED_EXAMPLES[1])
# random.shuffle(baseline_samples)
#
# # Then fine-tune with the same parameters:
# # model=meta-llama/Meta-Llama-3-8B-Instruct, lr=1e-5, epochs=20, full fine-tune